# MNIST Autoencoder (Dimensionality Reduction & Latent Space)

In this tutorial, we will use **Nanograd** to build and train a **Fully Connected Autoencoder** on MNIST digits. An autoencoder is trained in an unsupervised manner to compress (encode) input images into a small, lower-dimensional bottleneck vector, and then reconstruct (decode) the original image from that vector.

Specifically, we will:
1. Define an Autoencoder that compresses $28\times28$ (784 dimensional) images into a **2D latent space**.
2. Train the model using the **Adam** optimizer and **MSE** loss.
3. Display reconstructed images side-by-side with the original input digits.
4. Plot the 2D latent space coordinates of different digits, showing how the network naturally groups similar classes together.

---

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import gzip
import numpy as np
import matplotlib.pyplot as plt

from nanograd import Tensor, relu
from nanograd.nn import Layer
from nanograd.optim import Adam
from nanograd.loss import MSE

np.random.seed(42)

## 1. Loading the MNIST Dataset

We will load the pre-downloaded MNIST images. We select 1024 training images and 256 test images, flattening the inputs from $28\times28$ to 784-dimensional vectors.

In [ ]:
def load_mnist_images(filename):
    with gzip.open(filename, 'rb') as f:
        magic, num, rows, cols = np.frombuffer(f.read(16), dtype=np.dtype('>i4'))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(num, 1, rows, cols).astype(np.float32) / 255.0

def load_mnist_labels(filename):
    with gzip.open(filename, 'rb') as f:
        magic, num = np.frombuffer(f.read(8), dtype=np.dtype('>i4'))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.astype(np.int64)

all_train = load_mnist_images('data/train-images-idx3-ubyte.gz')
all_labels = load_mnist_labels('data/train-labels-idx1-ubyte.gz')
all_test = load_mnist_images('data/t10k-images-idx3-ubyte.gz')
all_test_labels = load_mnist_labels('data/t10k-labels-idx1-ubyte.gz')

train_size = 1024
test_size = 256

# Flatten images: (N, 1, 28, 28) -> (N, 784)
X_train = all_train[:train_size].reshape(train_size, 784)
y_train = all_labels[:train_size]

X_test = all_test[:test_size].reshape(test_size, 784)
y_test = all_test_labels[:test_size]

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)

## 2. Defining the Autoencoder Model

Our network compresses $784$ features down to a 2D bottleneck space: 
$$784 \xrightarrow{\text{FC1}} 64 \xrightarrow{\text{FC2}} 2 \xrightarrow{\text{FC3}} 64 \xrightarrow{\text{FC4}} 784$$

We manually override the linear outputs of the encoding layer (`fc2`) and reconstruction layer (`fc4`) to have identity activation.

In [ ]:
class Autoencoder:
    def __init__(self):
        # Encoder: 784 -> 64 -> 2
        self.encoder_fc1 = Layer(num_neurons=64, num_inputs=784)
        self.encoder_fc2 = Layer(num_neurons=2, num_inputs=64, activation_function=lambda x: x)
        
        # Decoder: 2 -> 64 -> 784
        self.decoder_fc1 = Layer(num_neurons=64, num_inputs=2)
        self.decoder_fc2 = Layer(num_neurons=784, num_inputs=64, activation_function=lambda x: x)
        
        # Apply He initialization to weights and zero to biases
        for layer in [self.encoder_fc1, self.encoder_fc2, self.decoder_fc1, self.decoder_fc2]:
            layer.weights.data = np.random.normal(0, np.sqrt(2.0 / layer.num_inputs), size=layer.weights.data.shape)
            layer.bias.data = np.zeros(layer.bias.data.shape)
            
    def encode(self, x: Tensor) -> Tensor:
        h = relu(self.encoder_fc1(x))
        z = self.encoder_fc2(h)
        return z
        
    def decode(self, z: Tensor) -> Tensor:
        h = relu(self.decoder_fc1(z))
        out = self.decoder_fc2(h)
        return out
        
    def __call__(self, x: Tensor) -> Tensor:
        return self.decode(self.encode(x))
        
    def params(self) -> list:
        return [self.encoder_fc1.weights, self.encoder_fc1.bias,
                self.encoder_fc2.weights, self.encoder_fc2.bias,
                self.decoder_fc1.weights, self.decoder_fc1.bias,
                self.decoder_fc2.weights, self.decoder_fc2.bias]

## 3. Training the Autoencoder

We train the model using **Adam** optimizer and **MSE** loss for 10 epochs. The target is the exact same image vector that was input to the network.

In [ ]:
model = Autoencoder()
optimizer = Adam(model.params(), learning_rate=0.005)
criterion = MSE()

epochs = 10
batch_size = 64

print("Starting training loop...")
for epoch in range(epochs):
    # Shuffle
    indices = np.arange(train_size)
    np.random.shuffle(indices)
    X_train_sh = X_train[indices]
    
    epoch_loss = 0.0
    num_batches = int(np.ceil(train_size / batch_size))
    
    for b in range(num_batches):
        start_idx = b * batch_size
        end_idx = min(start_idx + batch_size, train_size)
        
        x_batch = Tensor(X_train_sh[start_idx:end_idx])
        
        # Reconstruction objective: predict input
        reconstruction = model(x_batch)
        loss = criterion(reconstruction, x_batch)
        epoch_loss += loss.data.item() * (end_idx - start_idx)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    epoch_loss /= train_size
    print(f"Epoch {epoch+1:2d}/{epochs} | Reconstruction MSE Loss: {epoch_loss:.5f}")

## 4. Visualizing Reconstructed Digits

Let's pass some test images through the autoencoder and display original vs. reconstructed outputs.

In [ ]:
# Pass test set through model
x_test_tensor = Tensor(X_test)
reconstructed_test = model(x_test_tensor).data

# Plot
n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(16, 4.5))
for i in range(n_show):
    # Original
    axes[0, i].imshow(X_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title("Original Input", fontsize=12, fontweight='bold')
        
    # Reconstructed
    axes[1, i].imshow(reconstructed_test[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title("Autoencoder Output", fontsize=12, fontweight='bold')

plt.suptitle("Autoencoder Image Reconstructions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Visualizing 2D Latent Space Clustering

Since our bottleneck layer compresses images to exactly 2 dimensions, we can plot these 2D coordinates as a scatter plot and color-code them based on their actual label (0-9). This shows how the unsupervised bottleneck naturally groups similar patterns in separate coordinates!

In [ ]:
# Extract 2D bottleneck representations
latent_coords = model.encode(x_test_tensor).data

# Plot scatter map
plt.figure(figsize=(10, 8))
scatter = plt.scatter(latent_coords[:, 0], latent_coords[:, 1], c=y_test, s=30, cmap='tab10', alpha=0.85, edgecolors='none')
plt.colorbar(scatter, label='Digit Class (0-9)')
plt.title('MNIST 2D Latent Space Clustering')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.grid(True, linestyle='--', alpha=0.25)
plt.show()